## Instalacion de dependencias

In [1]:
!pip install langchain langchain-community langchain-huggingface \
             sentence-transformers pandas matplotlib scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Imports de dependencias

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

## Carga de chunks del notebook anterior:

Se carga el archivo `chunks_reglamento.csv` generado en el notebook 01.
Contiene todos los fragmentos del reglamento listos para vectorizar.

In [8]:
df_chunks = pd.read_csv('chunks_reglamento.csv')
textos = df_chunks['texto'].tolist()

print(f'chunks cargados: {len(textos)}')
print(f'Ejemplo de chunk cargado #0 \n{textos[40][:300]}')

chunks cargados: 695
Ejemplo de chunk cargado #0 
l) proponer al decanato, para su aprobación por el consejo de facultad, el otorgamiento 
de año sabático a los profesores. Las propuestas requieren de la aprobación en reunión 
de departamento, previo informe de la oficina de recursos humanos; 
m) evaluar permanentemente al personal a su cargo y coo


## Comparacion de configuraciones:

**Configuracion A:**
- nomic-embed-text (768 dims)
- Entrenado para tareas de RAG
- Mayor dimension y mayor capacidad de captura de matices

**Configuracion B:**
- B-BAAI/bge-small (384 dims)
- Ideal para documentos bilingues (ingles y español)
- Menor dimension pero mas rapido


In [9]:
print('Cargando Configuracion A')
t0 = time.time()
embedder_A  = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1",
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True}
)
print(f"  Config A lista en {time.time()-t0:.1f}s")


print('Cargando configuracion B')
t0 = time.time()
embedder_B = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}
)
print(f"  Config B lista en {time.time()-t0:.1f}s")

Cargando Configuracion A


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.3k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Config B lista en 1.9s


## Generacion de embeddings de una muestra

Se generan embeddings para una muestra de 50 chunks (no todos) para ahorrar
tiempo en Colab y poder comparar velocidad y dimensión entre ambos modelos.

In [10]:
muestra = 50
muestra_textos = textos[:muestra]

print(f'generando embeddings para {muestra} chunks')

t0 = time.time()
vecs_A = embedder_A.embed_documents(muestra_textos)
tiempo_A = time.time() - t0
print(f"Config A: {len(vecs_A)} vectores de dim {len(vecs_A[0])} en {tiempo_A:.2f}s")

t0 = time.time()
vecs_B = embedder_B.embed_documents(muestra_textos)
tiempo_B = time.time() - t0
print(f"Config B: {len(vecs_B)} vectores de dim {len(vecs_B[0])} en {tiempo_B:.2f}s")

vecs_A = np.array(vecs_A)
vecs_B = np.array(vecs_B)

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


generando embeddings para 50 chunks
Config A: 50 vectores de dim 768 en 132.45s
Config B: 50 vectores de dim 384 en 30.16s


## Comparacion utilizando preguntas de prueba

Se evalúan 5 preguntas típicas de estudiantes sobre el reglamento.
Para cada pregunta se calculan los 3 chunks más similares con cada modelo,
usando similitud coseno como métrica de comparación.

In [11]:
preguntas_prueba = [
    "¿Cuántas inasistencias me desaprueban el curso?",
    "¿Cuáles son los requisitos para sustentar la tesis?",
    "¿Cómo solicito convalidación de cursos?",
    "¿Qué pasa si desapruebo el mismo curso tres veces?",
    "¿Cuál es el promedio mínimo para mantener la beca?"
]

In [12]:
def recuperar_top3(pregunta, embedder, vectores, textos):
    vec_pregunta = np.array(embedder.embed_query(pregunta))
    sims = cosine_similarity([vec_pregunta], vectores)[0]
    top3_idx = sims.argsort()[-3:][::-1]
    return [(i, sims[i], textos[i][:150]) for i in top3_idx]

In [14]:
print("="*60)
for pregunta in preguntas_prueba:
    print(f"\nPREGUNTA: {pregunta}---------------------")
    print("\n  [Config A - nomic-embed-text]")
    for idx, sim, texto in recuperar_top3(pregunta, embedder_A, vecs_A, muestra_textos):
        print(f"    chunk #{idx} | sim={sim:.4f} | {texto}...")
    print("\n  [Config B - bge-small]")
    for idx, sim, texto in recuperar_top3(pregunta, embedder_B, vecs_B, muestra_textos):
        print(f"    chunk #{idx} | sim={sim:.4f} | {texto}...")
    print("-"*60)


PREGUNTA: ¿Cuántas inasistencias me desaprueban el curso?---------------------

  [Config A - nomic-embed-text]
    chunk #29 | sim=0.5906 | b) cumplir y hacer cumplir las disposiciones reglamentarias de la UNALM, los acuerdos 
del consejo de facultad y el consejo universitario en lo que le...
    chunk #45 | sim=0.5716 | de sus objetivos; 
d) informar regularmente al decano y al consejo de facultad sobre la situación 
académica, administrativa y contable de las maestrí...
    chunk #43 | sim=0.5711 | aprendizaje. 
w) todas las demás que señala el reglamento de la facultad respectiva. 
 
ARTÍCULO 20°.- La unidad de posgrado es la encargada de integr...

  [Config B - bge-small]
    chunk #40 | sim=0.6947 | l) proponer al decanato, para su aprobación por el consejo de facultad, el otorgamiento 
de año sabático a los profesores. Las propuestas requieren de...
    chunk #27 | sim=0.6826 | académicos son los mencionados en el artículo 19° del estatuto. Las facultades y 
departamentos acad

## Desicion de uso de embedding:

Debido al tiempo de inferencia y dimension del vector nos quedaremos con la configuracion A (nomic-embeded-text), ademas por su diseño especifico para tareas de RAG.

In [15]:
print(f"Generando embeddings de los {len(textos)} chunks completos...")

t0 = time.time()
todos_los_vectores = embedder_A.embed_documents(textos)
tiempo_total = time.time() - t0

todos_los_vectores = np.array(todos_los_vectores)
print(f"Vectores generados: {todos_los_vectores.shape}")
print(f"Tiempo total:       {tiempo_total:.1f}s")

np.save('vectores_reglamento.npy', todos_los_vectores)
print("\nGuardado: vectores_reglamento.npy")
print("Embeddings completos. Listo para 03_vector_db.ipynb")

Generando embeddings de los 695 chunks completos...
Esto puede tomar unos minutos en Colab...

Vectores generados: (695, 768)
Tiempo total:       1071.3s

Guardado: vectores_reglamento.npy
✅ Embeddings completos. Listo para 03_vector_db.ipynb
